# 第二问：动手建模前，先对齐三个关键认识

在正式建立数学模型之前，这个 notebook 先用数据核对三个基本判断：

1. 附件里的数据，能不能直接当作“每天 0 点制定计划时已经知道的量”；
2. 储能电池的容量和功率，够不够吸收净负荷的全部波动；
3. 历史数据里有没有可利用的规律，让日前预测成为可能。

这一轮只做数据核对，不求解购电策略。全年统计属于事后描述，不能当作 0 点决策时可用的预测输入。

数据来源：项目根目录的 `ProblemC/C题.md`、附件1、附件2 和 result2 模板；符号约定见 `notation.md`；可复用的读取与统计逻辑在 `utils/DataExplore/problem2/context_data.py`。

## 分析1：输入和输出的时间与单位是否一致

建模前先做最基础的体检：矩阵形状对不对、日期是否连续、有没有缺失值、结果模板的时段标签长什么样。

这一步不能省。功率乘以 1/6 小时才是一个时段的电量，单位换算错了，后面所有结论都会跟着错；时段标签错移一位，计费时刻和指定时刻的输出就会错位。所以这里不能只确认“文件能读进来”就完事。

In [2]:
!pip install openpyxl

Looking in indexes: https://mirrors.sustech.edu.cn/pypi/simple
  Using cached https://mirrors.sustech.edu.cn/pypi/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached https://mirrors.sustech.edu.cn/pypi/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [1]:
from pathlib import Path
import json
import numpy as np
from utils.DataExplore.problem2.context_data import inspect_sources, load_inputs, audit_matrix, energy_summary, causal_baselines
root = Path.cwd().parent
print(json.dumps(inspect_sources(root), ensure_ascii=False, default=str, indent=2))
typical, actual = load_inputs(root)
print("附件1列名：", typical.columns.tolist())
for name, frame in actual.items():
    print(name, json.dumps(audit_matrix(frame), ensure_ascii=False))

{
  "附件1.xlsx": {
    "Sheet1": {
      "shape": [
        145,
        4
      ],
      "top_left": [
        [
          "时间",
          "电价",
          "小区负载",
          "光伏发电预测功率"
        ],
        [
          "00:10:00",
          0.4248,
          3439.8466,
          0
        ],
        [
          "00:20:00",
          0.4245,
          3437.9792,
          0
        ],
        [
          "00:30:00",
          0.4269,
          3444.3031,
          0
        ]
      ],
      "top_right": [
        [
          "电价",
          "小区负载",
          "光伏发电预测功率"
        ],
        [
          0.4248,
          3439.8466,
          0
        ]
      ],
      "bottom_left": [
        [
          "23:50",
          0.4271,
          3439.061,
          0
        ],
        [
          "0:00+1",
          0.4276,
          3444.7259,
          0
        ]
      ]
    }
  },
  "附件2.xlsx": {
    "小区负载": {
      "shape": [
        366,
        145
      ],
      "top_left": [
        [
    

核对结果如下。

附件2的两张实际功率矩阵都是 365×144，日期连续、无重复，缺失值、非有限值、负值均为 0。数据很干净，暂时没有插补或剔除异常的依据。附件1有 144 条记录，负载峰值 7978.8849 kW，光伏峰值 10216.2 kW。

时段标签存在口径不一致，这是本轮发现的主要问题。附件的标签从 00:10 排到 0:00+1；结果模板从 0:10–0:20 排到 0:00–0:10+1；而 `notation.md` 按 0:00–0:10 到 23:50–24:00 理解。三套口径不完全相同。正式建模前必须确定：输入的点值究竟代表哪个 10 分钟区间、结果如何映射到模板。不能只按列顺序一一对应，就当作物理时刻已经对齐。本 notebook 只做对整天列移位不敏感的电量统计，暂时绕开这个问题。

计划购电模板共 335 行，去掉 1 行表头，对应 2月1日至12月31日的 334 天。1月的数据可以用来形成初始历史，这是合理解释，而非题面明说的规则；另外还要处理 1月1日的 6000 kWh 初始储能电量如何传递到 2月1日。

## 分析2：净负荷与储能的规模关系

电池同时受两种限制：能量容量限制它能存多少电，充放电功率限制它每个时段能进出多少电。把负载减去光伏得到净负荷，就能看出系统什么时候缺电、什么时候光伏有富余可以转移。

这里按“每条记录代表一个 10 分钟区间的功率”这一离散化假设统计电量，并顺带核对附件1典型日与全年均值的关系，避免把全年汇总特征误当成历史里可以预知的规律。

In [2]:
load = actual["小区负载"]
pv = actual["光伏发电实际功率"]
assert load.index.equals(pv.index) and load.columns.equals(pv.columns)
from utils.DataExplore.problem2.context_data import label_minutes
assert label_minutes(typical.iloc[:, 0]) == label_minutes(load.columns)
assert label_minutes(load.columns) == list(range(10, 1441, 10))
stats, selected = energy_summary(load, pv)
print(json.dumps(stats, ensure_ascii=False, indent=2))
print(selected.round(2).to_string())
print("固定电价范围：", typical["电价"].min(), typical["电价"].max())
print("典型日负载与全年同刻均值最大差_kW：", np.max(np.abs(typical["小区负载"].to_numpy() - load.mean().to_numpy())))
print("典型日光伏与全年同刻均值最大差_kW：", np.max(np.abs(typical["光伏发电预测功率"].to_numpy() - pv.mean().to_numpy())))

{
  "平均日负载_kWh": 111024.8080830137,
  "平均日光伏_kWh": 55482.8570183105,
  "平均日净用电_kWh": 55541.95106470319,
  "光伏盈余时段数": 10130,
  "光伏盈余时段占比": 0.19273211567732115,
  "最大盈余功率_kW": 6601.9911,
  "盈余超过5000kW的时段数": 241
}
               负载_kWh    光伏_kWh   净用电_kWh  盈余时段数
2025-03-20  117648.38  55439.42  62208.96     25
2025-06-21   81408.06  62073.32  19334.74     51
2025-09-23  121182.28  59065.67  62116.62     26
2025-12-21  124376.44  35338.66  89037.78      0
固定电价范围： 0.3713 1.3952
典型日负载与全年同刻均值最大差_kW： 5.0410958920110716e-05
典型日光伏与全年同刻均值最大差_kW： 0.04310438356164384


全年平均下来，每天负载 111024.81 kWh，光伏发电 55482.86 kWh，净用电 55541.95 kWh。全年有 10130 个时段光伏超过负载，占 19.27%；其中 241 个时段的盈余功率超过 5000 kW 的充放电功率上限，最大达到 6601.99 kW。在不允许售电、没有其他消纳通道、5000 kW 按母线侧理解的前提下，这些时段只靠电池无法吸收全部光伏盈余。

电池的可用储能跨度为 9600 kWh。按既定单向效率 0.9 计算，从满到空一次最多能向母线释放 8640 kWh，约为平均日净用电量的 15.56%。注意，这只是单次放电容量的比较，不是电池全天累计吞吐的上限。所以分析储能问题时，功率、容量、跨时段安排三者必须同时考虑。

四个指定日的日净用电分别约为 62209、19335、62117、89038 kWh；冬至日完全没有光伏盈余，夏至日有 51 个盈余时段。每天重复不变的是电价曲线，范围 0.3713–1.3952 元/kWh，而不是全天一个固定电价。

附件1的负载与全年同一时刻均值的最大差约 0.0000504 kW，光伏约 0.0431 kW。两者高度接近，但光伏的差异不能严格归结为四位小数舍入。更重要的是，第二问只指定使用附件1的电价；如果不加说明就把附件1的负载、光伏当作预测基线，相当于悄悄使用了未来信息。下面的预测分析只用过去日期的数据。

## 分析3：历史信息能否提供可检验的预测基线

对 2月1日至12月31日的每一天，分别用三种简单方法预测同一时刻的记录：前一日的值、前一周同刻的值、过去 7 日同刻的均值。所有预测都只使用当天之前的数据，不偷看未来。

这一步只回答“历史里有没有结构、还剩多少不确定性”，不负责挑选最终预测器。误差用 WAPE 衡量，即绝对误差之和除以实际值绝对值之和。净负荷允许为负，所以分母取绝对值之和，而不是净电量的代数和。最后用“负载取上周同刻、光伏取过去 7 日均值”的组合基线，检查相邻时段的残差是否相关，以此判断“各时段误差相互独立”的假设是否值得怀疑。

In [3]:
baselines, error_info = causal_baselines(load, pv)
print(baselines.round(4).to_string(index=False))
print(json.dumps(error_info, ensure_ascii=False, indent=2))

 对象       方法   MAE_kW  WAPE_pct
 负载    前一日同刻 650.0201   14.0795
 负载    前一周同刻 176.7957    3.8294
 负载 过去7日同刻均值 781.2111   16.9211
 光伏    前一日同刻 185.3465    7.7914
 光伏    前一周同刻 214.6882    9.0249
 光伏 过去7日同刻均值 153.3997    6.4485
净负荷    前一日同刻 744.2248   24.7997
净负荷    前一周同刻 327.7708   10.9223
净负荷 过去7日同刻均值 794.9682   26.4907
{
  "组合基线净负荷_MAE_kW": 273.35125210115484,
  "组合基线净负荷_WAPE_pct": 9.108864032033049,
  "日内相邻误差相关系数": 0.8497099732406003,
  "测试天数": 334
}


在 334 天严格只用日前历史的回测中：负载取上周同刻的 WAPE 为 3.8294%，明显优于前一日同刻的 14.0795%；光伏取过去 7 日均值的 WAPE 为 6.4485%，优于前一日的 7.7914%。分别预测负载和光伏再相减得到净负荷，WAPE 为 9.1089%，MAE 为 273.35 kW。结论是：历史数据存在可利用的结构，但预测误差无法被消灭，必须进入决策考量。误差量本身也不等于紧急购电量，后者还取决于计划安排与储能控制。

组合基线的净负荷残差，日内相邻时段的相关系数高达 0.8497，说明预测偏差往往连续出现，而非互相独立。这个汇总相关性只是探索性证据，不等于完成了独立同分布检验；它可能同时混有日级偏差和时段结构。后续若要构建随机模型，应当显式检验或保留时序相关性，不能未经验证就逐时段独立抽样。

还要提醒一点：这些数字是拿完全年数据回测后的比较结果，不能声称在 2月1日就已经知道哪种基线全年最优。如果要据此确定最终模型，需要明确划分验证期与测试期，或者滚动选择模型。本轮没有调参，也没有生成正式策略。

## 小结：形式化讨论前的共同起点

**本轮的工作方式。** 按照 `Q2Project.md` 的要求，先识别问题、明确符号和假设，再形式化并探索算法；数据探索服务于理解，本身不是目的。本轮沿用 `notation.md` 的约定，即电量按母线侧计量、充放电单向效率取 0.9；队友 `myh/solution.md` 中的算法建议仅作参考，不视为已定方案。

**题面明确给出的部分。** 每天 0 点制定当天计划；电价曲线每天重复同一条；缺电时按当时电价的 5 倍紧急购电；常规购电一律按计划量计费；设备受容量、功率、效率约束；正式结果覆盖 2—12月。物理上的电力缺口最终必须补齐，不能把“失供电”当成一种交钱就能了事的软约束。

**有依据但仍属解释的部分。** 附件2是实际发生过的轨迹，应当作为历史学习和逐时回放的数据；它在文件里看得见，不代表当天 0 点就能预知全天。第二问不引入附件3的光伏预报。“计划购电量在 0 点一次定死、储能调度在日内实时反馈”是合理的信息结构候选，但需要在讨论中明确写下来。当前时段的测量值是在控制决策之前、之中还是之后可用，也需要说明。如果允许在线响应，每个动作只能依赖已经揭示的历史和当前允许观测的信息。

**形式化之前必须拍板的事项。**

1. 信息与决策时点：哪些量在 0 点一次承诺，哪些量逐时响应。两阶段场景模型如果允许第二阶段看完整全天轨迹再调度电池，相当于预知未来，不能自动当成可执行的反馈策略。
2. 跨日状态：第二问没有重申日循环条件。合理的候选是“次日日初承接前日日末”；题目给出的初始值是 1月1日的 6000 kWh。需要说明 1月如何运行、2月1日的初始状态从哪里来、每天结束时剩余的电量如何计入规划，不能默认每天重置。
3. 能量与结算：现有符号 G、E、C、D 是不同阶段的决策量或实现量，S 是状态量，L、R 是外生实现量，不能全部当作已知参数罗列。如果计划购电视为足额交付，W 需要能容纳“计划多买了却用不掉”的电量；如果允许少取但照付，就应区分计划量与实际取电量。无售电、允许弃光或弃置剩余、充放电互斥，这些都要明确。
4. 目标口径：题面要求节省费用，但期望费用、风险惩罚、最坏情况费用是三种不同的建模选择，不能不加说明地自动加入。紧急购电单价已是 5 倍，如果已按 5 倍单独计费，不能再对同一笔紧急电量重复加收 1 倍基础费。
5. 时间口径：输入点标签、数学区间、输出模板必须统一。当前只完成了格式规范化，尚未裁定 10 分钟偏移问题。

**建议的下一步讨论顺序。** 先确定信息结构，再划分外生量、决策量与状态量，写出可行性条件和结算关系，最后才考察与经典问题的对应。储能的时间耦合特性和反馈策略的可实施性，是对齐经典问题时必须保留的结构，不能为了套用现成模型而丢掉。

**执行说明。** 本 notebook 使用本机已有的 Miniconda Python 与 nbformat/nbclient 执行，因为应用捆绑运行时缺少 notebook 依赖。原始 Excel 只读，未改动核心工作文档或统一符号。三个分析代码单元均保存真实执行结果。本轮未求解策略、未填写 result2。